# Session 2 Project

#### Create the vector database holding the boro regulations


In [ ]:
import re
import chromadb
import pypdf
from pathlib import Path

In [ ]:
SECTION_PATTERN = re.compile(
    r'(?=(?:Chapter\s+[A-Z]?\d+|ARTICLE\s+[IVXLC]+|§\s*[A-Z]?\d+[-\w]*\.))',
    re.MULTILINE
)
OVERLAP_CHARS = 200

def extract_sections(pdf_path: Path) -> list[dict]:
    reader = pypdf.PdfReader(pdf_path)
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    raw_chunks = SECTION_PATTERN.split(full_text)
    raw_chunks = [c.strip() for c in raw_chunks if c.strip()]

    sections = []
    for i, chunk in enumerate(raw_chunks):
        overlap = raw_chunks[i - 1][-OVERLAP_CHARS:] if i > 0 else ""
        sections.append({
            "text": overlap + chunk,
            "source": pdf_path.name,
            "chunk_index": i,
        })
    return sections

In [ ]:
client = chromadb.EphemeralClient()
collection = client.create_collection("boro_regulations")

data_dir = Path("data")
all_sections = []
for pdf_path in sorted(data_dir.glob("*.pdf")):
    all_sections.extend(extract_sections(pdf_path))

collection.add(
    documents=[s["text"] for s in all_sections],
    metadatas=[{"source": s["source"], "chunk_index": s["chunk_index"]} for s in all_sections],
    ids=[f"{s['source']}_chunk_{s['chunk_index']}" for s in all_sections],
)
print(f"Added {len(all_sections)} chunks from {len(list(data_dir.glob('*.pdf')))} PDFs")